# Pareto init weights — GAM analysis

Generalized additive models on **`pareto_results_init_2000samples/pareto_results.csv`**: smooth effects of log₁₀(λ) on `toya_r_squared` and `obj_homeo`.


### GAM: smooth effects of log₁₀(λ) on outcomes

A **Generalized Additive Model** fits an independent smooth spline `f(λᵢ)` for each weight, giving us the *shape* of how each λ drives the outcome — linear, saturating, threshold, etc.
We fit two GAMs (one per outcome) on the four log₁₀(λ) predictors and plot:
- **Partial dependence plots** — the marginal smooth `fᵢ(λᵢ)` for each predictor (others held at mean)
- **Summary table** — effective degrees of freedom and p-value per smooth (higher EDF = more nonlinear)

`pygam` uses penalised B-splines and selects smoothing strength via GCV automatically.


In [5]:
import os

import altair as alt
import numpy as np
import pandas as pd
from pygam import LinearGAM, s

os.chdir(os.path.expanduser("~/dev/vEcoli"))

CSV_PATH = (
    "notebooks/Heena notebooks/Metabolism_New Genes/"
    "pareto_results_init_2000samples/pareto_results.csv"
)
pareto_results = pd.read_csv(CSV_PATH)
print(len(pareto_results), "rows")
pareto_results.head()

2000 rows


,lambda_sec,lambda_eff,lambda_kin,lambda_div,obj_total,obj_homeo,obj_kin,obj_eff,obj_sec,obj_div,toya_pearson_r_squared,toya_r_squared
0,0.000210,2.349243e-05,0.000732,0.000066,0.019117,1.202514e-04,16.184786,195.054327,12.246192,0.004528,0.709356,-5.915941
1,0.000021,6.103205e-07,0.000027,0.000003,0.000833,2.462737e-08,22.790641,149.410636,6.312013,0.004792,0.716727,0.579211
2,0.000377,6.484861e-05,0.002555,0.000189,0.055031,1.783963e-04,11.332569,270.962783,22.097895,0.004480,0.695593,-34.306524
3,0.000124,4.010314e-06,0.000009,0.000004,0.001032,6.292688e-05,26.789582,134.262038,1.485147,0.005249,0.778458,0.514590
4,0.000002,2.908007e-08,0.000142,0.000002,0.001601,2.469378e-08,10.762509,302.489873,30.886056,0.004276,0.680590,-52.645270


In [6]:
GAM_LAMBDA_COLS = ["lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"]
GAM_OUTCOMES = ["toya_r_squared", "obj_homeo"]
GAM_PRETTY = {
    "lambda_sec": "λ_sec",
    "lambda_eff": "λ_eff",
    "lambda_kin": "λ_kin",
    "lambda_div": "λ_div",
}

_df = pareto_results[GAM_LAMBDA_COLS + GAM_OUTCOMES].copy()
_df[GAM_LAMBDA_COLS] = _df[GAM_LAMBDA_COLS].apply(np.log10)
_df = _df.replace([np.inf, -np.inf], np.nan).dropna()
X_full = _df[GAM_LAMBDA_COLS].to_numpy(dtype=float)


def smooth_edf(gam: LinearGAM, term_idx: int) -> float:
    """Sum of effective dof over B-spline coefficients for one smooth term."""
    st = sum(gam.terms[j].n_coefs for j in range(term_idx))
    nc = gam.terms[term_idx].n_coefs
    sl = slice(st, st + nc)
    return float(gam.statistics_["edof_per_coef"][sl].sum())


_gam_models: dict[str, LinearGAM] = {}
_summary_rows: list[dict] = []

for outcome in GAM_OUTCOMES:
    y = _df[outcome].to_numpy(dtype=float)
    X = X_full
    gam = LinearGAM(s(0) + s(1) + s(2) + s(3), n_splines=20).gridsearch(X, y)
    _gam_models[outcome] = gam

    y_hat = gam.predict(X)
    ss_res = float(np.sum((y - y_hat) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    pseudo_r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    print("\n" + "=" * 60)
    print(f" Outcome: {outcome}    pseudo-R² = {pseudo_r2:.4f}")
    print("=" * 60)
    gam.summary()

    pvals = gam.statistics_["p_values"]
    for i, lam in enumerate(GAM_LAMBDA_COLS):
        _summary_rows.append(
            {
                "outcome": outcome,
                "predictor": lam,
                "edf": smooth_edf(gam, i),
                "p_value": float(pvals[i]),
                "pseudo_r2_model": pseudo_r2,
            }
        )

summary_df = pd.DataFrame(_summary_rows)
display(summary_df.pivot(index="predictor", columns="outcome", values="edf").round(3))
display(summary_df.pivot(index="predictor", columns="outcome", values="p_value").round(4))

  0% (0 of 11) |                         | Elapsed Time: 0:00:00 ETA:  --:--:--
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: divide by zero encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: overflow encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: invalid value encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: divide by zero encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: overflow encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.1


 Outcome: toya_r_squared    pseudo-R² = 0.7443
LinearGAM                                                                                                 
=============================================== ==========================================================
Distribution:                        NormalDist Effective DoF:                                     19.8878
Link Function:                     IdentityLink Log Likelihood:                                 -7682.6884
Number of Samples:                         2000 AIC:                                            15407.1522
                                                AICc:                                           15407.6145
                                                GCV:                                              130.6863
                                                Scale:                                             11.3291
                                                Pseudo R-Squared:                               

/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: divide by zero encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: overflow encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: invalid value encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: divide by zero encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: overflow encountered in matmul
  B = (u @ vh[:rank]).conj().T
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/scipy/linalg/_basic.py:1622: RuntimeWarning: invalid value encou


 Outcome: obj_homeo    pseudo-R² = 0.7301
LinearGAM                                                                                                 
=============================================== ==========================================================
Distribution:                        NormalDist Effective DoF:                                     55.5232
Link Function:                     IdentityLink Log Likelihood:                                  18093.714
Number of Samples:                         2000 AIC:                                           -36074.3816
                                                AICc:                                          -36071.0339
                                                GCV:                                                   0.0
                                                Scale:                                                 0.0
                                                Pseudo R-Squared:                                   0

/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_97108/3191945572.py:41: UserWarning: KNOWN BUG: p-values computed in this summary are likely much smaller than they should be. 
 
Please do not make inferences based on these values! 

Collaborate on a solution, and stay up to date at: 
github.com/dswah/pyGAM/issues/163 

  gam.summary()


outcome,obj_homeo,toya_r_squared
predictor,,
lambda_div,13.511,4.633
lambda_eff,13.673,4.750
lambda_kin,13.616,4.687
lambda_sec,14.723,5.818


outcome,obj_homeo,toya_r_squared
predictor,,
lambda_div,0.3674,0.9977
lambda_eff,0.0000,0.0000
lambda_kin,0.0000,0.0000
lambda_sec,0.0000,0.0000


### Partial dependence (line + 95% CI) and raw data (subsample)

`generate_X_grid` returns a single matrix `(n_grid, n_features)` — assign it to one variable, do not unpack like `XX, _ = ...`.

Partial dependence and raw scatter use different y-axes (model effect vs measured outcome), so they are shown as two faceted charts.


In [7]:
_N_GRID = 100
_pdp_rows: list[dict] = []
_scatter_rows: list[dict] = []
_rng = np.random.default_rng(0)
_sample_idx = _rng.choice(len(_df), size=min(500, len(_df)), replace=False)

for outcome in GAM_OUTCOMES:
    gam = _gam_models[outcome]
    for i, lam in enumerate(GAM_LAMBDA_COLS):
        XX = gam.generate_X_grid(term=i, n=_N_GRID)
        pd_list = gam.partial_dependence(term=i, X=XX, width=0.95)
        pdep, conf = pd_list[0], np.asarray(pd_list[1])
        x_vals = XX[:, i]
        ci_lo, ci_hi = conf[:, 0], conf[:, 1]
        for xi, yi, lo, hi in zip(x_vals, pdep, ci_lo, ci_hi):
            _pdp_rows.append(
                {
                    "outcome": outcome,
                    "predictor": GAM_PRETTY[lam],
                    "x": float(xi),
                    "y": float(yi),
                    "ci_lo": float(lo),
                    "ci_hi": float(hi),
                }
            )
        for idx in _sample_idx:
            _scatter_rows.append(
                {
                    "outcome": outcome,
                    "predictor": GAM_PRETTY[lam],
                    "x": float(_df[lam].iloc[idx]),
                    "y_raw": float(_df[outcome].iloc[idx]),
                }
            )

pdp_df = pd.DataFrame(_pdp_rows)
scatter_df = pd.DataFrame(_scatter_rows)

# Layer first, then facet (Altair does not allow layering already-faceted charts)
_band = (
    alt.Chart(pdp_df)
    .mark_area(opacity=0.25, color="#E45756")
    .encode(
        x=alt.X("x:Q", title="log₁₀(λ)"),
        y=alt.Y("ci_lo:Q", title="partial effect"),
        y2="ci_hi:Q",
    )
)
_line = (
    alt.Chart(pdp_df)
    .mark_line(strokeWidth=2.2, color="#E45756")
    .encode(
        x=alt.X("x:Q", title="log₁₀(λ)"),
        y=alt.Y("y:Q", title="partial effect"),
    )
)
pdp_chart = (
    alt.layer(_band, _line)
    .properties(width=200, height=180, title="GAM partial dependence + 95% CI")
    .facet(
        row=alt.Row("outcome:N", title=None, header=alt.Header(labelAngle=0)),
        column=alt.Column("predictor:N", title=None),
    )
)

scatter_chart = (
    alt.Chart(scatter_df)
    .mark_circle(size=10, opacity=0.15, color="#4C78A8")
    .encode(
        x=alt.X("x:Q", title="log₁₀(λ)"),
        y=alt.Y("y_raw:Q", title="outcome"),
        row=alt.Row("outcome:N", title=None, header=alt.Header(labelAngle=0)),
        column=alt.Column("predictor:N", title=None),
    )
    .properties(width=200, height=180, title="Raw data (500 points subsample)")
)

pdp_chart & scatter_chart

alt.VConcatChart(...)